In [1]:
import pandas as pd
from pathlib import Path
import pandas as pd
import spacy

import plotly.express as px
import plotly.graph_objects as go
from PIL import Image

LUNAR_NAME_ALIASES = {
    "sea of tranquility": "mare tranquillitatis",
    "sea of tranquillity": "mare tranquillitatis",
    "sea of crises": "mare crisium",
}

In [2]:


def get_gazetteer_path(filename="moon_geolocation.csv"):
    """
    Dynamically constructs the absolute path to a file located in the
    same directory as the current running environment (Jupyter Notebook or script).
    """
    # Path(".") targets the current working directory where the Jupyter kernel is running
    base_dir = Path(".").resolve()
    
    # Combine the directory with the file name
    full_path = base_dir / filename
    
    # Safety Check: Let the user know if they forgot to drop the file there
    if not full_path.exists():
        print(f"⚠️ Warning: '{filename}' was not found at: {full_path}")
        print("Please verify the file name and that it is placed in this folder.")
    else:
        print(f"✅ Found gazetteer at: {full_path}")
        
    return full_path



In [3]:


def setup_lunar_parser(gazetteer_path):
    """
    Loads the USGS Moon Gazetteer and injects custom moon entities
    directly into spaCy's RoBERTa transformer pipeline.
    """
    # 1. Load the USGS Moon Gazetteer
    df = pd.read_csv(gazetteer_path)
    df.columns = df.columns.str.strip().str.replace(r"\s+", "_", regex=True)
    df['match_name'] = df['Feature_Name'].str.strip().str.casefold()
    
    # 2. Load the state-of-the-art English Transformer pipeline
    # The packaged tokenizer raises OverflowError with this Python 3.13/spaCy combination.
    nlp = spacy.load("en_core_web_trf", exclude=["tokenizer"])
    
    # 3. Create patterns from the gazetteer data
    patterns = []
    for row in df.itertuples(index=False):
        name = row.match_name
        if pd.isna(name):
            continue
        patterns.append({"label": "LUNAR_LOC", "pattern": name})
        feature_kind = str(row.Feature_Type).split(",", maxsplit=1)[0].strip().casefold()
        if feature_kind and not name.endswith(feature_kind):
            patterns.append({"label": "LUNAR_LOC", "pattern": f"{name} {feature_kind}"})
    for alias in LUNAR_NAME_ALIASES:
        patterns.append({"label": "LUNAR_LOC", "pattern": alias})
        
    # 4. Apply gazetteer matches after NER so they retain the lunar label.
    ruler = nlp.add_pipe(
        "entity_ruler",
        after="ner",
        config={"overwrite_ents": False, "phrase_matcher_attr": "LOWER"},
    )
    ruler.add_patterns(patterns)
    
    return nlp, df

def parse_lunar_locations(text, nlp_model, gazetteer_df):
    """
    Extracts lunar landmarks from text and matches them to coordinates.
    """
    doc = nlp_model(text)
    vehicle_entities = [entity for entity in doc.ents if entity.label_ == "PRODUCT"]
    gazetteer_by_alias = {}
    for row in gazetteer_df.to_dict("records"):
        name = row["match_name"]
        aliases = {name}
        feature_kind = str(row["Feature_Type"]).split(",", maxsplit=1)[0].strip().casefold()
        if feature_kind and not name.endswith(feature_kind):
            aliases.add(f"{name} {feature_kind}")
        for alias in aliases:
            gazetteer_by_alias[alias] = row

    for alias, official_name in LUNAR_NAME_ALIASES.items():
        match = gazetteer_df[gazetteer_df["match_name"] == official_name]
        if not match.empty:
            gazetteer_by_alias[alias] = match.iloc[0].to_dict()

    results = []
    for ent in doc.ents:
        candidate = ent.text.strip().casefold()
        if candidate.startswith("the "):
            candidate = candidate[4:]
        row = gazetteer_by_alias.get(candidate)
        if row is None:
            continue
        sentence_vehicles = [
            vehicle for vehicle in vehicle_entities
            if vehicle.sent.start == ent.sent.start
        ]
        vehicle_name = "Not identified"
        if sentence_vehicles:
            vehicle = min(sentence_vehicles, key=lambda item: abs(item.start_char - ent.start_char))
            vehicle_name = vehicle.text
            descriptor = next(
                (token.text for token in doc[vehicle.end:min(vehicle.end + 2, vehicle.sent.end)]
                 if token.lower_ in {"rover", "spacecraft", "lander", "module"}),
                None,
            )
            if descriptor:
                vehicle_name = f"{vehicle_name} {descriptor}"
        results.append({
            "text": ent.text,
            "official_name": row["Feature_Name"],
            "vehicle": vehicle_name,
            "feature_type": row["Feature_Type"],
            "latitude": row["Center_Latitude"],
            "longitude": row["Center_Longitude"],
            "start_char": ent.start_char,
            "end_char": ent.end_char,
        })
    return results


    


In [4]:
news_path = Path("nasa_news.txt")
if not news_path.is_file():
    raise FileNotFoundError(f"News text file not found: {news_path.resolve()}")
import re
news_text = news_path.read_text(encoding="utf-8").strip()
nasa_news = [entry.strip() for entry in re.split(r"(?m)^\s*#\s*$", news_text) if entry.strip()]
if not nasa_news:
    raise ValueError(f"News text file is empty: {news_path.resolve()}")

In [5]:
gazetteer_path = get_gazetteer_path("moon_geolocation.csv")
nlp, df = setup_lunar_parser(gazetteer_path)
locations = []
for entry_number, news_entry in enumerate(nasa_news, start=1):
    entry_locations = parse_lunar_locations(news_entry, nlp, df)
    for location in entry_locations:
        location["news_entry"] = entry_number
    locations.extend(entry_locations)
#print(locations)

✅ Found gazetteer at: /workspace/moon_geolocation.csv


/usr/local/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#print(locations)

In [7]:


map_df = pd.DataFrame(locations)
if map_df.empty:
    raise ValueError("No lunar locations were found to plot.")
map_df["entry_label"] = map_df["news_entry"].map(lambda entry: f"News entry {entry}")
map_df["place_vehicle_label"] = map_df.apply(
    lambda row: f"{row['official_name']} ({row['vehicle']})",
    axis=1,
)

fig = px.scatter(
    map_df,
    x="longitude",
    y="latitude",
    color="entry_label",
    hover_name="place_vehicle_label",
    text="place_vehicle_label",
    hover_data={"text": True, "vehicle": True, "news_entry": True, "latitude": ":.2f", "longitude": ":.2f"},
    labels={
        "longitude": "Longitude (degrees)",
        "latitude": "Latitude (degrees)",
        "entry_label": "News entry",
        "text": "Mention in text",
        "vehicle": "Vehicle in article",
        "news_entry": "News entry",
    },
    title="Lunar locations mentioned in the sample text",
)
fig.update_traces(marker={"size": 13, "line": {"width": 1, "color": "white"}}, textposition="top center", textfont={"size": 10, "color": "white"})

gazetteer_map = df.dropna(subset=["Feature_Name", "Center_Latitude", "Center_Longitude"]).copy()
gazetteer_map["Center_Latitude"] = pd.to_numeric(gazetteer_map["Center_Latitude"], errors="coerce")
gazetteer_map["Center_Longitude"] = pd.to_numeric(gazetteer_map["Center_Longitude"], errors="coerce")
gazetteer_map = gazetteer_map.dropna(subset=["Center_Latitude", "Center_Longitude"])
fig.add_trace(
    go.Scattergl(
        x=gazetteer_map["Center_Longitude"],
        y=gazetteer_map["Center_Latitude"],
        text=gazetteer_map["Feature_Name"],
        customdata=gazetteer_map["Feature_Type"],
        mode="markers+text",
        textposition="top center",
        textfont={"size": 8, "color": "#ffffff"},
        marker={"size": 3, "color": "#90EE90", "opacity": 0.75},
        name="All gazetteer place names (toggle)",
        visible="legendonly",
        hovertemplate="<b>%{text}</b><br>%{customdata}<br>Lon: %{x:.2f}<br>Lat: %{y:.2f}<extra></extra>",
    )
)
surface_image = "lunar_surface.jpg"
if not Path(surface_image).exists():
    raise FileNotFoundError(f"Lunar surface image not found: {Path(surface_image).resolve()}")

# The NASA texture is equirectangular: longitude maps to x, latitude to y.
fig.add_layout_image(
    dict(
        source=Image.open(surface_image),
        xref="x",
        yref="y",
        x=-180,
        y=90,
        sizex=360,
        sizey=180,
        sizing="stretch",
        opacity=0.9,
        layer="below",
    )
)
fig.update_xaxes(range=[-180, 180], dtick=30, showgrid=True, gridcolor="rgba(255,255,255,0.35)", zeroline=False)
fig.update_yaxes(range=[-90, 90], dtick=15, showgrid=True, gridcolor="rgba(255,255,255,0.35)", zeroline=False, scaleanchor="x", scaleratio=1)
fig.update_layout(
    width=1100,
    height=650,
    paper_bgcolor="#171b22",
    plot_bgcolor="#111111",
    font={"color": "#f3f4f6"},
    legend_title_text="News entry",
    margin={"l": 70, "r": 30, "t": 80, "b": 60},
)
fig.show()